<a href="https://colab.research.google.com/github/interpretune/examples/blob/main/tutorials/publish/tuning/circuit_tracer_tuning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Tuning a Circuit-Tracer Composition: Base Model, Drift, and Re-fitted Transcoders

A circuit-tracer composition runs a language model beside a transcoder for every MLP, and attribution graphs are drawn
through the transcoders. The transcoders were trained on the published model's activations, so fine-tuning the model
moves what they are asked to reconstruct. This notebook does both halves in one schedule and measures each step:

| Stage | Phase entry | What it tunes |
|---|---|---|
| 0 | `role:base & layers:-4:` | the base model's top four blocks on RTE, under the frozen transcoders |
| 1 | `role:latent:transcoder` | those layers' transcoders, promoted from `artifact`, re-fitted to the tuned model's MLPs, with the blocks refrozen |

Between stages, `TranscoderFitMeter` measures how well each layer's transcoder still reconstructs its MLP, and an
attribution graph of one prompt says what the drift costs: how much influence runs through error nodes rather than
features.

The schedule then runs a second time with the decoders **loaded lazily** (read from disk when needed, which is what lets
attribution fit on a small accelerator), tuned through low-rank corrections instead of whole.

The model is gemma-3-270m-it with its 16k-feature Gemma Scope 2 transcoders, small enough for a CPU.

Requirements: `uv pip install -e ".[examples,tune]" --group git-deps` (CPU torch is fine).

> `google/gemma-3-270m-it` is gated: accept its license on the Hugging Face model page and set
> `HF_GATED_PUBLIC_REPO_AUTH_KEY` (or `HF_TOKEN`). Until the `speediedan/rte` component is public, the same token also
> needs access to it. The transcoders are public (about 1.5 GB). No GPU is needed.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
LIMIT_TRAIN_BATCHES = 16  # batches per epoch; each stage is one epoch
LIMIT_VAL_BATCHES = 8
FIT_BATCHES = 4  # validation batches the transcoders' fit is measured on
TUNED_LAYERS = 4  # how many layers, from the top, each stage tunes
REFIT_LR = 1e-3  # stage 1's learning rate; stage 0 uses the configuration's
DELTA_RANK = 8  # rank of the second run's corrections to the lazily loaded decoders; None skips the second run
ATTRIBUTE = True  # draw an attribution graph before stage 0 and after each stage
SEED = 0
REVISION = (
    None  # rte component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [3]:
# @title Imports { display-mode: "form" }
import gc
import os
import re
import warnings

import torch
from circuit_tracer.graph import compute_graph_scores

import interpretune as it
from interpretune.adapters.circuit_tracer.drift import TranscoderFitMeter
from interpretune.config import SessionRunnerCfg
from interpretune.harness.notebook_ui import display_notebook_table
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.sites import resolve_site
from interpretune.utils import ITInstantiationFeedbackWarning

# the configuration classes are composed as declared; this feedback repeats their whole contents for each session
warnings.filterwarnings("ignore", category=ITInstantiationFeedbackWarning)

~/worktrees/interpretune-pub/src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html


~/worktrees/interpretune-pub/src/interpretune/analysis/ops/dispatcher.py:1242: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
~/worktrees/interpretune-pub/src/interpretune/analysis/ops/dispatcher.py:1242: Alias 'model_forward' already has an assigned op or alias so the alias specified by 'model_fwd' cannot be mapped to it


## 1. The Session

The session is the `rte_ft.gemma3.circuit_tracer+nnsight` configuration of the `speediedan/rte` component, on
`(core, nnsight, circuit_tracer)`. Three of its settings matter here:

- `tune_base: true` puts the base model's parameters in the composition's tunable view, with the role `base`. Without it
  a schedule naming them is refused.
- `lazy_encoder: false` and `lazy_decoder: false` hold the transcoders in memory as parameters, so they can be tuned
  whole. The second run sets `lazy_decoder` and a `lazy_delta_rank` instead.
- everything is fp32, and the run says so (`precision="32"`).

Two methods are added to the component's module:

- **`mlp_activations`** runs the model on a batch and returns every layer's transcoder input and MLP output, at the
  positions circuit-tracer attributes (padding and the chat template's first four tokens are left out). The sites are
  venngram addresses, `point:mlp.hook_in` and `point:hook_mlp_out`, resolved on this composition.
- **`training_step`** uses the component's RTE loss until a transcoder is thawed. From then on it trains the thawed
  transcoders to reconstruct the tuned model's MLP outputs: the loss is the fraction of variance unexplained, averaged
  over the thawed layers. The answer loss could not train them: the base model's forward does not run through its
  transcoders, so they receive no gradient from it.

In [4]:
# @title 1: The session { display-mode: "form" }
CONFIG = "rte_ft.gemma3.circuit_tracer+nnsight"
SKIPPED_POSITIONS = 4  # the template's `<bos><start_of_turn>user\n`, which circuit-tracer leaves out of its graphs

if REVISION is None:  # a validation run passes a local-publish REVISION and needs none of these steps
    it.hub.pull("speediedan/rte")  # caches the component's manifest and entrypoint; executes nothing
    os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running its entrypoint, for this session only
    it.hub.pull("speediedan/rte", CONFIG)


def build_session(delta_rank: int | None) -> it.ITSession:
    itdm_cfg, it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", CONFIG, revision=REVISION)
    if delta_rank is not None:
        it_cfg.circuit_tracer_cfg.lazy_decoder = True
        it_cfg.circuit_tracer_cfg.lazy_delta_rank = delta_rank

    class TunedModule(m_cls):
        @property
        def transcoders(self):
            return self.hf_model.transcoders.transcoders  # one per layer

        def mlp_activations(self, batch) -> tuple[torch.Tensor, torch.Tensor]:
            """Each layer's transcoder input and MLP output, shaped (layers, positions, d_model)."""
            batch = self.batch_to_device(batch)  # as the core loop does: the dataloader yields CPU tensors
            seen: dict[tuple[str, int], torch.Tensor] = {}
            handles = []
            for layer in range(len(self.transcoders)):
                for point in ("mlp.hook_in", "hook_mlp_out"):
                    site = resolve_site(self, f"point:{point} & layers:{layer}")
                    assert site.io == "output", site

                    def record(module, args, output, key=(point, layer)) -> None:
                        seen[key] = output.detach()

                    handles.append(self.get_submodule(site.path).register_forward_hook(record))
            try:
                with torch.no_grad():
                    self(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            finally:
                for handle in handles:
                    handle.remove()
            mask = batch["attention_mask"].bool()
            keep = mask & (mask.cumsum(dim=1) > SKIPPED_POSITIONS)
            inputs, outputs = (
                torch.stack([seen[point, layer][keep] for layer in range(len(self.transcoders))])
                for point in ("mlp.hook_in", "hook_mlp_out")
            )
            return inputs, outputs

        def training_step(self, batch, batch_idx):
            thawed = [i for i, tc in enumerate(self.transcoders) if any(p.requires_grad for p in tc.parameters())]
            if not thawed:
                return super().training_step(batch, batch_idx)
            inputs, outputs = self.mlp_activations(batch)
            unexplained = []
            for layer in thawed:
                y = outputs[layer]
                error = y - self.transcoders[layer](inputs[layer])
                unexplained.append(error.pow(2).sum() / (y - y.mean(0)).pow(2).sum())
            loss = torch.stack(unexplained).mean()
            self.log("train_loss", loss)
            return loss

    session = it.ITSession(
        it.ITSessionConfig(
            adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
            datamodule_cfg=itdm_cfg,
            module_cfg=it_cfg,
            datamodule_cls=dm_cls,
            module_cls=TunedModule,
        )
    )
    it.it_init(**session)
    return session

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

## 2. The Schedule

```python
{
    0: {"params": ["role:base & layers:-4:"], "max_transition_epoch": 1},
    1: {
        "params": ["role:latent:transcoder"],
        "promote": {"role:artifact & layers:-4:": "latent:transcoder"},
        "refreeze": ["role:base & layers:-4:"],
        "lr": 1e-3,
    },
}
```

Every entry is an address. The base model and the transcoders share their layers, so the role says which is meant:
`role:base & layers:-4:` is the top four blocks, and `role:artifact & layers:-4:` is those layers' transcoders, which a
schedule never thaws until something promotes them. Stage 1 promotes them to `latent:transcoder`, thaws them and holds
the blocks where stage 0 left them.

At three points (as published, after stage 0 and after stage 1) the run measures:

- each layer's **fit**, with `TranscoderFitMeter` over a few validation batches: the fraction of variance unexplained
  (FVU) of the MLP outputs, and the mean number of active features per position ($L_0$);
- an **attribution graph** of one prompt, reduced to circuit-tracer's two scores. The replacement score is the share of
  influence from tokens to logits that runs through features rather than error nodes. The completeness score is the
  influence-weighted share of every node's inputs that are not error nodes.

Attribution runs with every thawed tensor frozen and leaves no gradient behind, so it is safe between two optimizer
steps. After stage 0 it warns that parameters outside the transcoders are thawed and suggests what this notebook does
next: measure the fit, and re-fit the transcoders if it has degraded.

In [5]:
# @title 2: Run the schedule, measuring between stages { display-mode: "form" }
TOP = f"layers:-{TUNED_LAYERS}:"
SCHEDULE = {
    0: {"params": [f"role:base & {TOP}"], "max_transition_epoch": 1},
    1: {
        "params": ["role:latent:transcoder"],
        "promote": {f"role:artifact & {TOP}": "latent:transcoder"},
        "refreeze": [f"role:base & {TOP}"],
        "lr": REFIT_LR,
    },
}
QUESTION = (
    "Dana sold the house in May. Does the previous passage imply that Dana sold a house?"
    " Answer with only one word, either Yes or No."
)


def measure(module) -> dict:
    meter = TranscoderFitMeter(module.hf_model.transcoders)
    for i, batch in enumerate(module.datamodule.val_dataloader()):
        if i >= FIT_BATCHES:
            break
        meter.update(*module.mlp_activations(batch))
    scores = None
    if ATTRIBUTE:
        prompt = module.datamodule.tokenizer.apply_chat_template(
            [{"role": "user", "content": QUESTION}], tokenize=False, add_generation_prompt=True
        )
        scores = compute_graph_scores(module.generate_attribution_graph(prompt))
        module.attribution_graphs.clear()
    return {"fit": meter.compute(), "scores": scores}


def run(delta_rank: int | None) -> dict:
    module = (session := build_session(delta_rank)).module
    torch.manual_seed(SEED)  # after the build, so both runs draw the same batches
    initial = {n: p.detach().clone() for n, p in module.named_parameters()}
    result = {"measures": {"as published": measure(module)}, "stages": {}}

    class RecordStages:
        def on_train_epoch_end(self, ctx, metrics) -> None:
            stage = fts.tuning_scheduler.curr_depth
            result["stages"][stage] = {
                "thawed": sorted(n for n, p in module.named_parameters() if p.requires_grad),
                "metrics": {k: float(v) for k, v in (metrics or {}).items()},
            }
            result["measures"][f"after stage {stage}"] = measure(module)

    fts = FTSCoreCallback(SCHEDULE)
    SessionRunner(
        run_cfg=SessionRunnerCfg(
            it_session=session,
            max_epochs=len(SCHEDULE),
            limit_train_batches=LIMIT_TRAIN_BATCHES,
            limit_val_batches=LIMIT_VAL_BATCHES,
            precision="32",
            callbacks=[RecordStages(), fts],  # the recorder first, so it reads each stage before a transition
        )
    ).train()
    assert fts.tuning_scheduler.curr_depth == len(SCHEDULE) - 1, "every stage must run"
    result["sizes"] = {n: p.numel() for n, p in module.named_parameters()}
    result["changed"] = sorted(n for n, p in module.named_parameters() if not torch.equal(initial[n], p))
    return result


RUNS = {"whole transcoders": run(None)}
if DELTA_RANK is not None:
    gc.collect()  # one session in memory at a time
    RUNS[f"rank-{DELTA_RANK} decoder corrections"] = run(DELTA_RANK)

~/worktrees/interpretune-pub/src/interpretune/utils/import_utils.py:223: Configured auth-token environment variable 'HF_GATED_PUBLIC_REPO_AUTH_KEY' is not set. Proceeding without an explicit token: any ambient Hugging Face credential (e.g. from `huggingface-cli login`) still applies, but access to gated resources will fail later if none exists. Export HF_GATED_PUBLIC_REPO_AUTH_KEY to use an explicit token.


[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


... (60 outputs left out)


Epoch 0 end: {'train_loss': 2.3103, 'val_loss': 0.8318, 'accuracy': 0.4375}



Epoch 1 end: {'train_loss': 0.2500, 'val_loss': 0.8318, 'accuracy': 0.4375}


[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule


## 3. What Each Stage Tuned, and What It Did to the Fit

The first table counts what each stage thawed. The second follows the fit of the tuned layers' transcoders through the
run, beside the layers below them, which neither stage touches. The third is the attribution graph's scores.

In [6]:
# @title 3: The stages, the fit and the graph { display-mode: "form" }
def kind(name: str) -> str:
    return "transcoder" if ".transcoders." in name else "base"


def layer_of(name: str) -> int:
    return int(re.search(r"\.(?:layers|transcoders)\.(\d+)\.", name).group(1))


def mean_fit(fit: list, layers: range, field: str) -> float:
    return sum(getattr(fit[layer], field) for layer in layers) / len(layers)


display_notebook_table(
    [
        [name, stage, f"`{', '.join(SCHEDULE[stage]['params'])}`", len(s["thawed"]), ", ".join(sorted({kind(n) for n in s["thawed"]})),
         f"{sum(r['sizes'][n] for n in s['thawed']) / 1e6:.2f} M", f"{s['metrics']['val_loss']:.4f}"]
        for name, r in RUNS.items()
        for stage, s in sorted(r["stages"].items())
    ],
    ["Run", "Stage", "Phase entry", "Thawed tensors", "Kind", "Thawed parameters", "RTE val_loss"],
    title="What each stage thawed",
)

n_layers = len(next(iter(RUNS.values()))["measures"]["as published"]["fit"])
tuned, below = range(n_layers - TUNED_LAYERS, n_layers), range(n_layers - TUNED_LAYERS)
display_notebook_table(
    [
        [name, when, f"{mean_fit(m['fit'], tuned, 'fvu'):.4f}", f"{mean_fit(m['fit'], tuned, 'l0'):.1f}",
         f"{mean_fit(m['fit'], below, 'fvu'):.4f}"]
        for name, r in RUNS.items()
        for when, m in r["measures"].items()
    ],
    ["Run", "When", f"FVU, top {TUNED_LAYERS} layers", "L0, same layers", "FVU, layers below"],
    title="How well the transcoders reconstruct the model's MLPs",
    caption=f"Mean over layers, {RUNS['whole transcoders']['measures']['as published']['fit'][0].positions} validation positions.",
)
if ATTRIBUTE:
    display_notebook_table(
        [
            [name, when, *(f"{score:.4f}" for score in m["scores"])]
            for name, r in RUNS.items()
            for when, m in r["measures"].items()
        ],
        ["Run", "When", "Replacement score", "Completeness score"],
        title="The attribution graph of one prompt",
        caption="Higher is more of the computation through features and tokens rather than error nodes.",
    )

for name, r in RUNS.items():
    s0, s1 = r["stages"][0], r["stages"][1]
    assert {kind(n) for n in s0["thawed"]} == {"base"}, f"{name}: stage 0 thaws the base model alone"
    assert {kind(n) for n in s1["thawed"]} == {"transcoder"}, f"{name}: stage 1 thaws the promoted transcoders alone"
    assert {layer_of(n) for n in [*s0["thawed"], *s1["thawed"]]} == set(tuned), f"{name}: only the top layers are tuned"
    assert set(r["changed"]) <= {*s0["thawed"], *s1["thawed"]}, f"{name}: nothing else changed"
    assert s0["metrics"]["val_loss"] == s1["metrics"]["val_loss"], f"{name}: stage 1 holds the base model"
    fits = [m["fit"] for m in r["measures"].values()]
    for fit in fits[1:]:
        assert all(abs(fit[i].fvu - fits[0][i].fvu) < 1e-6 for i in below), f"{name}: the layers below do not drift"
    published, drifted, refitted = (mean_fit(fit, tuned, "fvu") for fit in fits)
    assert drifted != published and refitted != drifted, f"{name}: each stage moves the fit"
    if LIMIT_TRAIN_BATCHES >= 16:  # a run of a few steps shows the mechanics, not a direction
        assert drifted > published and refitted < drifted, f"{name}: tuning the base degrades the fit; re-fitting helps"
if len(RUNS) > 1:
    whole, lowrank = RUNS.values()
    assert [f.fvu for f in whole["measures"]["after stage 0"]["fit"]] == [
        f.fvu for f in lowrank["measures"]["after stage 0"]["fit"]
    ], "a lazily loaded decoder with zero corrections is the same transcoder, and the base is tuned identically"

Run,Stage,Phase entry,Thawed tensors,Kind,Thawed parameters,RTE val_loss
whole transcoders,0,`role:base & layers:-4:`,52,base,22.29 M,0.8318
whole transcoders,1,`role:latent:transcoder`,24,transcoder,85.66 M,0.8318
rank-8 decoder corrections,0,`role:base & layers:-4:`,52,base,22.29 M,0.8318
rank-8 decoder corrections,1,`role:latent:transcoder`,28,transcoder,44.26 M,0.8318


Run,When,"FVU, top 4 layers","L0, same layers","FVU, layers below"
whole transcoders,as published,0.2086,21.2,0.0768
whole transcoders,after stage 0,0.2368,18.0,0.0768
whole transcoders,after stage 1,0.2048,18.0,0.0768
rank-8 decoder corrections,as published,0.2086,21.2,0.0768
rank-8 decoder corrections,after stage 0,0.2368,18.0,0.0768
rank-8 decoder corrections,after stage 1,0.2307,18.0,0.0768


Run,When,Replacement score,Completeness score
whole transcoders,as published,0.7739,0.9247
whole transcoders,after stage 0,0.7388,0.9094
whole transcoders,after stage 1,0.7639,0.9191
rank-8 decoder corrections,as published,0.7739,0.9247
rank-8 decoder corrections,after stage 0,0.7388,0.9094
rank-8 decoder corrections,after stage 1,0.7450,0.9118


**Reading the tables.**

- **Each stage tuned what its address says.** Stage 0 thawed the top four blocks' base tensors; stage 1 thawed only
  those layers' transcoders. The RTE loss does not move in stage 1: the base model is refrozen, and its forward does not
  run through the transcoders.
- **Tuning the base moved the transcoders' fit.** After stage 0 the top layers' unexplained variance is higher, fewer
  features are active, and the graph's scores are lower: more of the prompt's computation runs through error nodes. The
  layers below are unchanged, since nothing below the tuned blocks moved.
- **Re-fitting brings it back.** Tuned whole, the top layers' transcoders reconstruct the tuned model's MLPs about as well
  as the published ones did the published model's, and the graph's scores recover most of what they lost. The second run
  tunes the same encoders whole but reaches its decoders only through rank-8 corrections, and recovers a smaller part.
- **The two runs tune the base identically.** A lazily loaded decoder with a zero correction is the same transcoder, so
  the fit after stage 0 is the same to the last bit.

Sixteen steps on four-example batches show the mechanics, not a result: how far a real fine-tune drifts, and what a
re-fit costs, are the questions of [interpretune#709](https://github.com/interpretune/interpretune/issues/709). The
[circuit-tracer tuning page](https://interpretune.org/en/latest/usage/tuning_circuit_tracer.html) describes the view,
the roles and the low-rank corrections.